#### Notebook para prever a máscara de instâncias dada uma imagem

In [ ]:
import torch
from torch import nn
import time
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torchvision.transforms as T
from skimage.measure import label
from skimage.segmentation import watershed

imagem =" data/stage1_train/0a7d30b252359a10fd298b638b90cb9ada3acced4e0c0e5a3692013f432ee4e9/images/0a7d30b252359a10fd298b638b90cb9ada3acced4e0c0e5a3692013f432ee4e9.png"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels) -> None:
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self,x):
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, in_channels, out_channels) -> None:
        super(UNet, self).__init__()
        # Encoder
        self.conv1 = DoubleConv(in_channels, 64)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = DoubleConv(64,128)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Bottleneck
        self.bottleneck = DoubleConv(128,256)

        # Decoder
        self.up1 = nn.ConvTranspose2d(256,128,kernel_size=2, stride=2)
        self.dec1 = DoubleConv(256,128)
        self.up2 = nn.ConvTranspose2d(128,64,kernel_size=2, stride=2)
        self.dec2 = DoubleConv(128,64)

        # Output
        self.final_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        # Passa pelo primeiro bloco e salva
        skip1 = self.conv1(x)
        x = self.pool1(skip1)
        skip2 = self.conv2(x)
        x = self.pool2(skip2)


        x = self.bottleneck(x)
        x = self.up1(x)
        x = torch.cat((skip2, x), dim=1)
        x = self.dec1(x)


        x = self.up2(x)
        x = torch.cat((skip1, x), dim=1)
        x = self.dec2(x)

        return self.final_conv(x)


def binary_decoder(mask, threshold):
    return (torch.sigmoid(mask) > threshold).float()

def load_best_model(path):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = UNet(in_channels=3, out_channels=3).to(device)
    path_weights = path
    weights = torch.load(path_weights, map_location=device, weights_only=True)
    model.load_state_dict(weights)
    model.eval()

    return model

model = load_best_model('melhor_modelo_parte3.pth')

ModuleNotFoundError: No module named 'skimage'

In [6]:
!pip install skimage

Defaulting to user installation because normal site-packages is not writeable
  Using cached skimage-0.0.tar.gz (757 bytes)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [3 lines of output]
      
      *** Please install the `scikit-image` package (instead of `skimage`) ***
      
      [end of output]
  
  note: This error originates from a subprocess, and is likely not a problem with pip.

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: C:\Users\walle\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [ ]:
def preprocess_image(image_path):
    # Carrega a imagem e garante que está em RGB
    img = Image.open(image_path).convert("RGB")
    
    # Aplica as mesmas transformações usadas no DataLoader de validação
    transform = T.Compose([
        T.Resize((256, 256)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # Adiciona a dimensão do batch [B, C, H, W]
    img_tensor = transform(img).unsqueeze(0) 
    return img, img_tensor

def apply_watershed_single(pred_class_numpy):
    # Isola o interior e as células completas
    interior_mask = (pred_class_numpy == 1)
    cell_pixels = (pred_class_numpy != 0)

    # Cria os marcadores a partir do interior
    markers, num_objects = label(interior_mask, return_num=True)

    # Aplica o Watershed
    image_blank = np.zeros_like(pred_class_numpy)
    instance_mask = watershed(image=image_blank, markers=markers, mask=cell_pixels)

    return instance_mask

def prever_imagem(model, caminho_imagem, device):
    # Preparação
    img_original, img_tensor = preprocess_image(caminho_imagem)
    img_tensor = img_tensor.to(device)

    # Inferência Semântica
    with torch.no_grad():
        pred_logits = model(img_tensor)
        # Tira o batch e joga pro Numpy
        pred_classes = torch.argmax(pred_logits, dim=1).squeeze(0).cpu().numpy() 

    # Pós-processamento de Instâncias
    mascara_instancias = apply_watershed_single(pred_classes)

    # Exibição dos Resultados
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    axes[0].imshow(img_original.resize((256, 256)))
    axes[0].set_title("Imagem Original")
    axes[0].axis('off')

    axes[1].imshow(pred_classes, cmap='nipy_spectral')
    axes[1].set_title("Predição Semântica (Rede)")
    axes[1].axis('off')

    axes[2].imshow(mascara_instancias, cmap='nipy_spectral')
    axes[2].set_title("Instâncias Finais (Watershed)")
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()
    
    return mascara_instancias

In [ ]:
mascara_final = prever_imagem(model, imagem, device)
mascara_final